In [1]:
# -------- Imports --------
import sys
import os
import numpy as np
import scipy.sparse as sp
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from _utility import *
from _fractures import *
from _initial_conditions import *

# -- Qiskit imports --
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import MatrixExponential
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

np.random.seed(0)

In [2]:
# -- Grid parameters --

# -- Lower/Upper Bound, Number of Points, Separation Between Points --

xmin,ymin,zmin,xmax,ymax,zmax,Nx,Ny,Nz,dx,dy,dz = get_grid_parameters()

# -- Staggered Grid Points for each wavefield component of seismic wave equation w = [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

N_main,N_vx,N_vy,N_vz,N_sxy,N_sxz,N_syz,N_vel,N_stress,psi_len = get_grid_size(Nx,Ny,Nz)

In [3]:
N_syz

80

In [4]:
#Define a function to get the indices of the points for the subspace energy calculation

def get_mask_indices(ith_element,wave_field_component):
    subspace_points = []
    for i in range(len(ith_element)):
        for j in ith_element[i]: 
            if(wave_field_component=='vx'):
                subspace_points.append(j)
            elif(wave_field_component=='vy'):
                subspace_points.append(N_vx+j)
            elif(wave_field_component=='vz'):
                subspace_points.append(N_vx+N_vy+j)
            elif(wave_field_component=='sxx'):
                subspace_points.append(N_vel+j)
            elif(wave_field_component=='syy'):
                subspace_points.append(N_vel+N_main+j)
            elif(wave_field_component=='szz'):
                subspace_points.append(N_vel+2*N_main+j)
            elif(wave_field_component=='sxy'): 
                subspace_points.append(N_vel+3*N_main+j)
            elif(wave_field_component=='sxz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+j)
            elif(wave_field_component=='syz'):
                subspace_points.append(N_vel+3*N_main+N_sxy+N_sxz+j)
            else:
                print("Not a valid wave field component")
                
    return list(filter(lambda x: x < psi_len, subspace_points))

In [9]:
#Define a function to return the positions of points in the subspace within each component (i location)

def get_i_location(Nx,Ny,Nz,z_min,z_max,wave_field_component):
    
    assert z_min >= 0 and z_max <= Nz-1 and z_min < z_max
    i_location = []
    #Should be 7 different unique amounts of grid points for the same z location, the main grid, σ_xy, σ_xz, σ_yz, v_x, v_y, and v_z
    
    #vx
    if(wave_field_component=='vx'):
        i_location.append(np.arange((Nx-1)*Ny*z_min,(Nx-1)*Ny*z_max,1))
    #vy
    elif(wave_field_component=='vy'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*z_max,1))
    #vz
    elif(wave_field_component=='vz'):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*(z_max),1))
    #main
    elif(wave_field_component=='sxx' or wave_field_component=='syy' or wave_field_component=='szz' ):
        i_location.append(np.arange(Nx*Ny*z_min,Nx*Ny*z_max,1))
    #stress_xy
    elif(wave_field_component=='sxy'):
        i_location.append(np.arange((Nx-1)*(Ny-1)*z_min,(Nx-1)*(Ny-1)*z_max,1))
    #stress_xz
    elif(wave_field_component=='sxz'):
        i_location.append(np.arange((Nx-1)*(Ny)*z_min,(Nx-1)*Ny*(z_max),1))
    #stress_yz
    elif(wave_field_component=='syz'):
        i_location.append(np.arange(Nx*(Ny-1)*z_min,Nx*(Ny-1)*(z_max),1))
    else:
        print('Not valid wave field component')

    #print(f'i_location {type(i_location)}: {i_location}')
    return i_location

In [10]:
# A function returning a dictionary of masks for kinetic, potential, and total energy subspace projectors
def threeProjectors(mask, Nx, Ny, Nz):
    kinetic = np.copy(mask)
    potential = np.copy(mask)
    kinetic_num = (Nx-1)*Ny*Nz + Nx*(Ny-1)*Nz + Nx*Ny*(Nz-1)
    kinetic[kinetic_num:] = 0
    potential[:kinetic_num] = 0
    return {
        'kinetic' : kinetic,
        'potential' : potential,
        'total' : mask
    }

In [13]:
# -------- Simulation --------
# Simulate the quantum circuit, calculate the expectation value of the energy in the subspace
def hamiltonian_simulation(qc,observable):
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, observable)], precision=1e-30)
    result = job.result()
    print(f" Layer > Expectation value: {result[0].data.evs}")
    print(f" > Metadata: {result[0].metadata}")
    return result

In [17]:
# Important: The following implementation of the material parameters is not scalable. This code is only for demonstration purposes and assumes oracle access. 
# In a real-world scenario, the material parameters and the Hamiltonian need to be sparsly constructed on the QC itself.
# Furthermore, this code uses direct matrix exponentiation, which is inefficient for large matrices. Other integration methods should be used (e.g. Trotter-Suzuki, Qubitization).


# Density model and S matrix (3D with one horizontal fracture)

#Set fracture thickness
fracture_thickness = 0
rho_model, S_matrix = two_perpendicular_fractures(Nx,Ny,Nz,dx,dy,dz,fracture_thickness)

# -------- Simulation (3D Elastic) (Oracle Access) --------
(H, A, _, B_sqrt, B_inv, _) = FD_solver_3D_elastic_quantum(Nx, Ny, Nz, dx, dy, dz, rho_model, S_matrix)

H_pauli = SparsePauliOp.from_operator(Operator(H.toarray())) # Expensive conversion
print("Hamiltonian shape: ", H.shape)
print("Hamiltonian NNZ-Ratio: ", H.nnz/H.shape[0]**2)
print("Pauli Terms (inefficient representation): ", len(H_pauli))

# Number of grid points
print("Number of grid points: ", psi_len)

# Number of qubits
n = (H.shape[0]-1).bit_length()
print("Number of qubits (for wave field): ", n)


Adding perpendicular fracture planes:
  Fracture: Vertical plane at x-index 2
  Fracture: Horizontal plane at z-index 2
  Total fracture cells: 10 out of 125 (8.0%)

Density Model Statistics:
  Min: 1000.0 kg/m³, Max: 2700.0 kg/m³
  Mean: 2564.0 kg/m³, Std: 461.2 kg/m³
Hamiltonian shape:  (1024, 1024)
Hamiltonian NNZ-Ratio:  0.0029754638671875
Pauli Terms (inefficient representation):  125320
Number of grid points:  915
Number of qubits (for wave field):  10


In [19]:
# -- Initial Conditions (3D Elastic) --
# State vector: [v_x, v_y, v_z, σ_xx, σ_yy, σ_zz, σ_xy, σ_xz, σ_yz]

phi_0 = gaussian_IC(Nx,Ny,Nz,dx,dy,dz,xmin,xmax,ymin,ymax,zmin,zmax)

# Pad the initial conditions with zeros to match Hamiltonian size (power of 2)
phi_0 = np.concatenate([phi_0, np.zeros(H.shape[0] - psi_len)])

# Normalize the initial state and transform it to the energy basis
psi_0 = B_sqrt @ phi_0
norm = np.linalg.norm(psi_0)
psi_0 /= norm

# Number of non-zero initial state values
psi_0_nnz = np.sum(psi_0 != 0)
print('Initial state NNZ-Ratio:', psi_0_nnz/psi_len)

Initial state NNZ-Ratio: 0.4098360655737705


In [21]:
def get_quantum_circuit(mask,t):
    synthesis = MatrixExponential()
    evo = PauliEvolutionGate(H_pauli, time=t, synthesis=synthesis, label='U=exp(-iH_real*t)')

    # -- Quantum Simulation --
    # Define the quantum registers
    reg_wave = QuantumRegister(n, 'wave')
    reg_P = QuantumRegister(1, 'P')

    # Initialize the quantum circuit
    qc = QuantumCircuit(reg_wave, reg_P)
    qc.prepare_state(psi_0, reg_wave)
    print("Normalization: ",np.linalg.norm(psi_0))
    # Apply the time-evolution operator
    qc.append(evo, reg_wave)

    # Apply unitary "projection" transform (Can often be combined into fewer MCX gates)
    qc.x(reg_P)
    for d in np.nonzero(mask)[0]:
        ctrl_bin = format(d, f'0{n}b')  # binary string of length n
        qc.mcx(reg_wave, target_qubit=reg_P, ctrl_state=ctrl_bin)

    # Define observable
    O1 = Pauli('I'*(n+1))
    O2 = Pauli('Z'+'I'*n) 
    observable = SparsePauliOp([O1, O2], [0.5, 0.5])

    return qc,observable

In [23]:
# Compute the energy loss (post-processing)
def get_quantum_subspace_energy(result):
    E = np.abs(result[0].data.evs)
    EN_QC = (1/2) * E * norm**2 * (dx * dy * dz)
    print('Expected value: ',E)
    print('Subspace Energy Estimate (quantum):', EN_QC.round(30))
    return EN_QC.round(30)

In [25]:
# Time Integration (Has classical integration errors (DOP853))
def get_classical_state_vector(t):
    #phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='DOP853').y.T[-1] 
    phi_t = solve_ivp(lambda t, phi: (B_inv @ A @ phi), (0, t), phi_0, t_eval=(0,t), method='RK45').y.T[-1] 
    psi_t_classical = B_sqrt @ phi_t
    return psi_t_classical

In [27]:
def get_classical_subspace_energy_1(mask,psi_t_classical):
    masked_state = np.diag(mask) @ psi_t_classical
    
    # Probability of being in the masked subspace
    prob_in_subspace = np.linalg.norm(masked_state)**2 / np.linalg.norm(psi_t_classical)**2
    
    print("prob in subspace",prob_in_subspace)
    
    EN_CL = (1/2) * prob_in_subspace * np.linalg.norm(psi_t_classical)**2 * (dx * dy * dz)
    print('Subspace Energy Estimate (classical 1):', EN_CL.round(30))
    return EN_CL.round(30)

In [29]:
def get_classical_subspace_energy(mask,psi_t_classical):

    EN_CL = (1/2) * np.linalg.norm(np.diag(mask) @ psi_t_classical, axis=0)**2 * (dx * dy * dz)
    print('Subspace Energy Estimate (classical 2):', EN_CL.round(30))
    return EN_CL.round(30)

In [55]:
## Test to see if the code is working by just defining one mask and one time 

#Define some layer to get the subspace for

layer_min = 3
layer_max = 4

test_mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vx'),'vx'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vy'),'vy'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'vz'),'vz'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxx'),'sxx'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'syy'),'syy'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'szz'),'szz'))
+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxy'),'sxy'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'sxz'),'sxz'))+subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min,layer_max,'syz'),'syz'))
test_mask = np.concatenate([test_mask, np.zeros(H.shape[0]-psi_len)])
test_time = 



SyntaxError: invalid syntax (2804118185.py, line 12)

In [231]:
## Get the test qc and observable 

qc,observable = get_quantum_circuit(test_mask,test_time)

Normalization:  0.9999999999999999


In [233]:
## Get the test result 

result_1 = hamiltonian_simulation(qc,observable)
#result_2 = hamiltonian_simulation_2(qc,observable)

 Layer > Expectation value: -8.22675261247241e-14
 > Metadata: {'target_precision': 1e-30, 'circuit_metadata': {}}


In [234]:
## Get the test quantum subspace energy

qc_sub_energy = get_quantum_subspace_energy(result_1)

Expected value:  8.22675261247241e-14
Subspace Energy Estimate (quantum): 9.024430446894262e-05


In [235]:
## Get the classical state vector

psi_t_classical = get_classical_state_vector(test_time)

In [236]:
## Get the first estimation of classical energy

cl_sub_energy_1 = get_classical_subspace_energy_1(test_mask,psi_t_classical)

cl_sub_energy_2 = get_classical_subspace_energy(test_mask,psi_t_classical)


prob in subspace 2.2359116831068835e-15
Subspace Energy Estimate (classical 1): 2.452708914451286e-06
Subspace Energy Estimate (classical 2): 2.4527089144512866e-06


In [31]:
## RUN THE SIMULATION FOR MULTIPLE LAYERS AND TIMES
num_layers = 4 #8
num_times = 5 #5
t = 1
layer_spacing = (Nz-1)//num_layers
times = np.linspace(1/num_times,t,num_times)
layer_min = np.arange(0,Nz-1,layer_spacing)
layer_max = layer_min+layer_spacing

## MASK DEFINITION FOR TOTAL, KINETIC, and POTENTIAL ENERGIES

masks = []
mask_layer = []
components = ['vx','vy','vz','sxx','syy','szz','sxy','sxz','syz']
for i in range(num_layers):
    mask_layer=[]
    mask_kinetic = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    mask_potential = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    mask_total = np.concatenate([np.zeros(psi_len),np.zeros(H.shape[0]-psi_len)])
    for j in range(len(components)):
        mask = subspaceProjector(psi_len,get_mask_indices(get_i_location(Nx,Ny,Nz,layer_min[i],layer_max[i],components[j]),components[j]))
        mask = np.concatenate([mask, np.zeros(H.shape[0]-psi_len)])
        mask_total+=mask
        if(components[j]=='vx' or components[j]=='vy' or components[j]=='vz'):
            print("kinetic")
            mask_kinetic+=mask
        else:
            print("elastic")
            mask_potential+=mask
    mask_layer.append(mask_kinetic)
    mask_layer.append(mask_potential)
    mask_layer.append(mask_total)
    masks.append(mask_layer)

kinetic
kinetic
kinetic
elastic
elastic
elastic
elastic
elastic
elastic
kinetic
kinetic
kinetic
elastic
elastic
elastic
elastic
elastic
elastic
kinetic
kinetic
kinetic
elastic
elastic
elastic
elastic
elastic
elastic
kinetic
kinetic
kinetic
elastic
elastic
elastic
elastic
elastic
elastic


array([1, 2, 3, 4])

In [138]:
counter = 0
for i in range(len(masks[0][0])):
    print(masks[1][2][i],counter+1)
    counter+=1

0.0 1
0.0 2
0.0 3
0.0 4
0.0 5
0.0 6
0.0 7
0.0 8
0.0 9
0.0 10
0.0 11
0.0 12
0.0 13
0.0 14
0.0 15
0.0 16
0.0 17
0.0 18
0.0 19
0.0 20
1.0 21
1.0 22
1.0 23
1.0 24
1.0 25
1.0 26
1.0 27
1.0 28
1.0 29
1.0 30
1.0 31
1.0 32
1.0 33
1.0 34
1.0 35
1.0 36
1.0 37
1.0 38
1.0 39
1.0 40
0.0 41
0.0 42
0.0 43
0.0 44
0.0 45
0.0 46
0.0 47
0.0 48
0.0 49
0.0 50
0.0 51
0.0 52
0.0 53
0.0 54
0.0 55
0.0 56
0.0 57
0.0 58
0.0 59
0.0 60
0.0 61
0.0 62
0.0 63
0.0 64
0.0 65
0.0 66
0.0 67
0.0 68
0.0 69
0.0 70
0.0 71
0.0 72
0.0 73
0.0 74
0.0 75
0.0 76
0.0 77
0.0 78
0.0 79
0.0 80
0.0 81
0.0 82
0.0 83
0.0 84
0.0 85
0.0 86
0.0 87
0.0 88
0.0 89
0.0 90
0.0 91
0.0 92
0.0 93
0.0 94
0.0 95
0.0 96
0.0 97
0.0 98
0.0 99
0.0 100
0.0 101
0.0 102
0.0 103
0.0 104
0.0 105
0.0 106
0.0 107
0.0 108
0.0 109
0.0 110
0.0 111
0.0 112
0.0 113
0.0 114
0.0 115
0.0 116
0.0 117
0.0 118
0.0 119
0.0 120
1.0 121
1.0 122
1.0 123
1.0 124
1.0 125
1.0 126
1.0 127
1.0 128
1.0 129
1.0 130
1.0 131
1.0 132
1.0 133
1.0 134
1.0 135
1.0 136
1.0 137
1.0 138
1.0 

In [33]:
## RUN THE SIMULATION FOR MULTIPLE TIMES

subspace_energies_total_quantum = []
subspace_energies_kinetic_quantum = []
subspace_energies_elastic_quantum = []
subspace_energies_total_classical = []
subspace_energies_kinetic_classical = []
subspace_energies_elastic_classical = []

time_subspace_energies_total_quantum = []
time_subspace_energies_kinetic_quantum = []
time_subspace_energies_elastic_quantum = []
time_subspace_energies_total_classical = []
time_subspace_energies_kinetic_classical = []
time_subspace_energies_elastic_classical = []

energy_type = ['kinetic','potential','total']

test_mask = masks[0] #Contains 9 masks for each component
qe_layer = 0
ce_layer = 0
for time in range(len(times)):
    #Only have to calculate classical state vector once per time
    psi_t_classical = get_classical_state_vector(times[time])
    subspace_energies_total_quantum = []
    subspace_energies_kinetic_quantum = []
    subspace_energies_elastic_quantum = []
    subspace_energies_total_classical = []
    subspace_energies_kinetic_classical = []
    subspace_energies_elastic_classical = []
    for j in range(len(masks)):
        qe_layer=0
        ce_layer=0
        qe_layer_k=0
        ce_layer_k=0
        qe_layer_e=0
        ce_layer_e=0
        layer_mask = masks[j]
        for k in range(len(layer_mask)):
            qe_mask = 0
            ce_mask = 0
            qc, observable = get_quantum_circuit(layer_mask[k],times[time])
            result = hamiltonian_simulation(qc,observable)
            qe_mask = get_quantum_subspace_energy(result)
            ce_mask = get_classical_subspace_energy(layer_mask[k],psi_t_classical)
            if(energy_type[k]=='kinetic'):
                qe_layer_k+=qe_mask
                ce_layer_k+=ce_mask
            elif(energy_type[k]=='potential'):
                qe_layer_e+=qe_mask
                ce_layer_e+=ce_mask
            else:
                qe_layer+=qe_mask
                ce_layer+=ce_mask
        print("Quantum Energy Layer: ",j," Energy Type (cumulative): ",energy_type[k]," ",qe_layer)
        print("Classical Energy Layer: ",j,"Energy Type (cumulative): ",energy_type[k]," ",ce_layer)

        subspace_energies_total_quantum.append(qe_layer)
        subspace_energies_total_classical.append(ce_layer)
        subspace_energies_kinetic_quantum.append(qe_layer_k)
        subspace_energies_kinetic_classical.append(ce_layer_k)
        subspace_energies_elastic_quantum.append(qe_layer_e)
        subspace_energies_elastic_classical.append(ce_layer_e)

    time_subspace_energies_total_quantum.append(subspace_energies_total_quantum)
    time_subspace_energies_total_classical.append(subspace_energies_total_classical)
    time_subspace_energies_kinetic_quantum.append(subspace_energies_kinetic_quantum)
    time_subspace_energies_kinetic_classical.append(subspace_energies_kinetic_classical)
    time_subspace_energies_elastic_quantum.append(subspace_energies_elastic_quantum)
    time_subspace_energies_elastic_classical.append(subspace_energies_elastic_classical)

Normalization:  0.9999999999999999
 Layer > Expectation value: -2.7489122089718876e-13
 > Metadata: {'target_precision': 1e-30, 'circuit_metadata': {}}
Expected value:  2.7489122089718876e-13
Subspace Energy Estimate (quantum): 1.7146668201735693e-09
Subspace Energy Estimate (classical 2): 1.2684936612793238e-12
Normalization:  0.9999999999999999
 Layer > Expectation value: -5.058176100192213e-13
 > Metadata: {'target_precision': 1e-30, 'circuit_metadata': {}}
Expected value:  5.058176100192213e-13
Subspace Energy Estimate (quantum): 3.1550977514987004e-09
Subspace Energy Estimate (classical 2): 2.3449287003581207e-11
Normalization:  0.9999999999999999
 Layer > Expectation value: -7.500666754367558e-13
 > Metadata: {'target_precision': 1e-30, 'circuit_metadata': {}}
Expected value:  7.500666754367558e-13
Subspace Energy Estimate (quantum): 4.678630467323358e-09
Subspace Energy Estimate (classical 2): 2.4717780664860532e-11
Quantum Energy Layer:  0  Energy Type (cumulative):  total   4.

In [37]:
time_subspace_energies_total_quantum = np.array(time_subspace_energies_total_quantum)
time_subspace_energies_total_classical = np.array(time_subspace_energies_total_classical)
time_subspace_energies_kinetic_quantum = np.array(time_subspace_energies_kinetic_quantum)
time_subspace_energies_kinetic_classical = np.array(time_subspace_energies_kinetic_classical)
time_subspace_energies_elastic_quantum = np.array(time_subspace_energies_elastic_quantum)
time_subspace_energies_elastic_classical = np.array(time_subspace_energies_elastic_classical)


In [41]:

import pandas as pd
import numpy as np
time_array=[]
for i in range(len(times)):
    for j in range(len(subspace_energies_total_quantum)):
        time_array.append(times[i])
        
total_quantum=time_subspace_energies_total_quantum.flatten()
total_classical=time_subspace_energies_total_classical.flatten()
kinetic_quantum=time_subspace_energies_kinetic_quantum.flatten() 
kinetic_classical=time_subspace_energies_kinetic_classical.flatten()
potential_quantum=time_subspace_energies_elastic_quantum.flatten()
potential_classical=time_subspace_energies_elastic_classical.flatten()

## CREATE PANDAS DATAFRAME TO EXPORT AS A CSV
data = {
    'total_quantum':total_quantum,
    'total_classical': total_classical,
    'kinetic_quantum': kinetic_quantum,
    'kinetic_classical': kinetic_classical,
    'potential_quantum': potential_quantum,
    'potential_classical': potential_classical,
    'time':time_array
}
df = pd.DataFrame(data)

file_name = 'energies_time_'+ str(0.2)+'_'+str(1)
df.to_csv(file_name, index=False)

In [43]:
df

,total_quantum,total_classical,kinetic_quantum,kinetic_classical,potential_quantum,potential_classical,time
0,4.678630e-09,2.471778e-11,1.714667e-09,1.268494e-12,3.155098e-09,2.344929e-11,0.2
1,5.136435e+03,5.136435e+03,2.397590e-08,2.538835e-08,5.136435e+03,5.136435e+03,0.2
2,1.096961e+03,1.096961e+03,1.493755e-09,3.553942e-09,1.096961e+03,1.096961e+03,0.2
3,4.191269e+00,4.191269e+00,1.664113e-09,2.703108e-11,4.191269e+00,4.191269e+00,0.2
4,4.740957e-09,2.853847e-11,1.732672e-09,1.433312e-12,3.211191e-09,2.710516e-11,0.4
5,5.136435e+03,5.136435e+03,1.010140e-07,1.015486e-07,5.136435e+03,5.136435e+03,0.4
6,1.096961e+03,1.096961e+03,1.106050e-08,1.421383e-08,1.096961e+03,1.096961e+03,0.4
7,4.191269e+00,4.191269e+00,1.606635e-09,1.054157e-10,4.191269e+00,4.191269e+00,0.4
8,4.558133e-09,2.507505e-11,1.556081e-09,1.534980e-12,3.035293e-09,2.354007e-11,0.6
9,5.136435e+03,5.136435e+03,2.295818e-07,2.284832e-07,5.136435e+03,5.136435e+03,0.6


NameError: name 'masks' is not defined

In [17]:
print(dx)

5.0


In [25]:
norm**2*dx*dy*dz

185.14871106052868

6237.619428431535